# 🏦 Sistema Inteligente de Análise de Perfil Financeiro
## Tema 3 — Risk Intelligence: Detecção de Risco em Transações Financeiras

> **Hackathon Artemisia · Elas Tech · Projeto de Dados**  
> Responsáveis pelo módulo de Machine Learning: **Jany & Karen**

---

### Objetivo
Desenvolver um modelo preditivo capaz de identificar transações financeiras de **risco elevado**, gerando um **score de risco** que alimentará o dashboard de alertas e projeções da equipe de dados.

### Fluxo do Módulo
```
CSV com target (pré-processado via PySpark + Parquet)
       ↓
 Amostragem Estratificada
       ↓
 Pré-Processamento & Feature Engineering
       ↓
 Balanceamento de Classes (SMOTE)
       ↓
 Treinamento: Random Forest + Gradient Boosting
       ↓
 Avaliação (AUC-ROC, Precision, Recall, F1)
       ↓
 Score de Risco por Transação
       ↓
 Exportação para o Power BI (CSV)
```

---
## 1. Importação de Bibliotecas

In [ ]:
# ── Manipulação de dados ────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualização ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Pré-processamento ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Balanceamento de classes ────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ── Modelos ─────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# ── Métricas de avaliação ───────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay,
    average_precision_score,
)

# ── Configurações gerais ────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

print('✅ Bibliotecas importadas com sucesso!')

---
## 2. Carregamento e Amostragem Estratificada

> **Contexto:** O dataset completo contém ~8,9 milhões de linhas. Para viabilizar o treinamento local sem perda de representatividade, realizamos uma **amostragem estratificada** que preserva a proporção original da variável `target` (≈0,15% de risco).

In [ ]:
# ─── Parâmetros de amostragem ───────────────────────────────────────────────
CAMINHO_CSV = 'ML_06.csv'   # ajuste o caminho conforme necessário
TAMANHO_AMOSTRA = 300_000   # linhas a carregar (ajuste conforme RAM disponível)

print(f'📂 Carregando amostra de {TAMANHO_AMOSTRA:,} linhas...')

# Carregamento inicial para contagem total
df_raw = pd.read_csv(CAMINHO_CSV, nrows=TAMANHO_AMOSTRA)

print(f'\n📊 Dimensões da amostra: {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas')
print('\n🔍 Distribuição do target (risco):')
dist = df_raw['target'].value_counts()
dist_pct = df_raw['target'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Contagem': dist, 'Percentual (%)': dist_pct.round(3)}))

In [ ]:
# ─── Separar classes e montar amostra equilibrada para inspeção ─────────────
df_risco     = df_raw[df_raw['target'] == 'Yes'].copy()
df_sem_risco = df_raw[df_raw['target'] == 'No'].copy()

n_risco = len(df_risco)
print(f'⚠️  Transações de risco (Yes): {n_risco:,}')
print(f'✅  Transações normais   (No):  {len(df_sem_risco):,}')

df = df_raw.copy()   # usamos a amostra completa para treino (balanceamento via SMOTE)
print('\n✅ Dataset pronto para pré-processamento.')

---
## 3. Análise Exploratória (EDA Focada em Risco)

> Esta seção gera visualizações que **serão reutilizadas pela colega responsável pelo storytelling no Power BI**.

In [ ]:
# ─── 3.1 Visão Geral do Dataset ─────────────────────────────────────────────
print('=== INFORMAÇÕES GERAIS DO DATASET ===')
print(f'Linhas:  {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]}')
print('\nTipos de dados:')
print(df.dtypes)
print('\nValores nulos por coluna:')
print(df.isnull().sum())
print('\nEstatísticas descritivas (numéricas):')
df.describe().T

In [ ]:
# ─── 3.2 Distribuição do Target ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
cores = ['#2ecc71', '#e74c3c']
contagens = df['target'].value_counts()
axes[0].bar(contagens.index, contagens.values, color=cores, edgecolor='white', linewidth=1.5)
for i, (idx, val) in enumerate(contagens.items()):
    axes[0].text(i, val + 500, f'{val:,}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Volume de Transações por Classe', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de Transações')
axes[0].set_xlabel('Classe (target)')

# Gráfico de pizza
pcts = df['target'].value_counts(normalize=True) * 100
axes[1].pie(
    pcts.values,
    labels=[f'Sem Risco\n({pcts["No"]:.2f}%)', f'Com Risco\n({pcts["Yes"]:.2f}%)'],
    colors=cores,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 11}
)
axes[1].set_title('Proporção de Risco Financeiro', fontsize=13, fontweight='bold')

plt.suptitle('Desbalanceamento de Classes — Risco Financeiro', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distribuicao_target.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: distribuicao_target.png')

In [ ]:
# ─── 3.3 Distribuição das Variáveis Numéricas por Classe ────────────────────
variaveis_num = ['valor_transacao', 'limite', 'porcentagem_limit_usad']
labels_pt = {
    'valor_transacao': 'Valor da Transação (R$)',
    'limite': 'Limite de Crédito (R$)',
    'porcentagem_limit_usad': 'Uso do Limite (%)'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, variaveis_num):
    # Limitar outliers extremos para visualização
    q1, q99 = df[col].quantile(0.01), df[col].quantile(0.99)
    dados_clip = df[df[col].between(q1, q99)]
    
    sns.kdeplot(
        data=dados_clip, x=col, hue='target',
        palette={'No': '#2ecc71', 'Yes': '#e74c3c'},
        fill=True, alpha=0.4, ax=ax
    )
    ax.set_title(labels_pt[col], fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(title='Risco', labels=['Sem Risco', 'Com Risco'])

plt.suptitle('Distribuição das Variáveis por Classe de Risco', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('distribuicao_variaveis.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: distribuicao_variaveis.png')

In [ ]:
# ─── 3.4 Boxplot: Valor da Transação vs Risco ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, variaveis_num):
    q99 = df[col].quantile(0.99)
    dados_clip = df[df[col] <= q99]
    
    sns.boxplot(
        data=dados_clip, x='target', y=col,
        palette={'No': '#2ecc71', 'Yes': '#e74c3c'},
        width=0.4, fliersize=2, ax=ax
    )
    ax.set_title(labels_pt[col], fontsize=12, fontweight='bold')
    ax.set_xlabel('Classe de Risco')
    ax.set_xticklabels(['Sem Risco', 'Com Risco'])

plt.suptitle('Comparativo por Classe de Risco (até percentil 99)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplot_variaveis.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: boxplot_variaveis.png')

In [ ]:
# ─── 3.5 Análise Temporal — Risco por Hora do Dia ───────────────────────────
df['data_trans'] = pd.to_datetime(df['data_trans'])
df['hora']       = df['data_trans'].dt.hour
df['dia_semana'] = df['data_trans'].dt.day_name()

# Taxa de risco por hora
risco_hora = (
    df.groupby('hora')['target']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .reset_index()
)
risco_hora.columns = ['hora', 'taxa_risco_pct']

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(
    risco_hora['hora'], risco_hora['taxa_risco_pct'],
    color=[
        '#e74c3c' if h in [0, 1, 2, 3, 4, 5, 23] else '#3498db'
        for h in risco_hora['hora']
    ],
    edgecolor='white', linewidth=0.8
)
ax.set_xlabel('Hora do Dia', fontsize=12)
ax.set_ylabel('Taxa de Risco (%)', fontsize=12)
ax.set_title('Taxa de Risco por Hora do Dia\n(🔴 Período noturno de alerta: 23h–05h)', fontsize=13, fontweight='bold')
ax.set_xticks(range(24))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f%%'))

# Legenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Horário noturno (risco elevado)'),
    Patch(facecolor='#3498db', label='Horário comercial')
]
ax.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.savefig('risco_por_hora.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: risco_por_hora.png')

---
## 4. Pré-Processamento e Feature Engineering

In [ ]:
# ─── 4.1 Tratamento de Nulos ─────────────────────────────────────────────────
print('Nulos antes do tratamento:')
print(df.isnull().sum())

# porcentagem_limit_usad: preencher com mediana (variável contínua com outliers)
mediana_plu = df['porcentagem_limit_usad'].median()
df['porcentagem_limit_usad'] = df['porcentagem_limit_usad'].fillna(mediana_plu)

print(f'\n✅ Nulos preenchidos com mediana: {mediana_plu:.4f}')
print('\nNulos após tratamento:')
print(df.isnull().sum())

In [ ]:
# ─── 4.2 Engenharia de Features (Feature Engineering) ───────────────────────
#
# Criamos variáveis derivadas que capturam comportamentos de risco:
# - Padrões temporais (hora, período do dia)
# - Indicadores de uso anormal do limite
# - Segmentos de valor de transação

# Já criamos 'hora' na etapa de EDA; garantir que existe
if 'hora' not in df.columns:
    df['data_trans'] = pd.to_datetime(df['data_trans'])
    df['hora'] = df['data_trans'].dt.hour

# Flag: transação noturna (23h–05h — período de maior risco)
df['flag_noturna'] = df['hora'].apply(lambda h: 1 if (h >= 23 or h <= 5) else 0)

# Flag: uso do limite acima de 80% (pressão financeira elevada)
df['flag_limite_alto'] = (df['porcentagem_limit_usad'] > 80).astype(int)

# Flag: valor da transação acima do percentil 95 (transação incomum)
p95_valor = df['valor_transacao'].quantile(0.95)
df['flag_valor_alto'] = (df['valor_transacao'] > p95_valor).astype(int)

# Flag: limite zerado ou muito baixo (< R$100) — perfil de risco
df['flag_limite_zero'] = (df['limite'] < 100).astype(int)

# Ratio: valor / limite (quando limite > 0)
df['ratio_valor_limite'] = np.where(
    df['limite'] > 0,
    df['valor_transacao'] / df['limite'],
    0
)

# Score composto de risco (soma das flags = 0 a 4)
df['score_risco_bruto'] = (
    df['flag_noturna'] +
    df['flag_limite_alto'] +
    df['flag_valor_alto'] +
    df['flag_limite_zero']
)

print('✅ Features criadas com sucesso!')
novas = ['flag_noturna','flag_limite_alto','flag_valor_alto','flag_limite_zero',
         'ratio_valor_limite','score_risco_bruto']
print(f'\nNovas variáveis: {novas}')
df[novas].describe().T

In [ ]:
# ─── 4.3 Correlação das Features com o Target ────────────────────────────────
df['target_num'] = (df['target'] == 'Yes').astype(int)

features_corr = [
    'valor_transacao', 'limite', 'porcentagem_limit_usad',
    'hora', 'flag_noturna', 'flag_limite_alto', 'flag_valor_alto',
    'flag_limite_zero', 'ratio_valor_limite', 'score_risco_bruto'
]

corr = df[features_corr + ['target_num']].corr()['target_num'].drop('target_num').sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
cores_corr = ['#e74c3c' if v > 0 else '#3498db' for v in corr.values]
ax.barh(corr.index, corr.values, color=cores_corr, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Correlação das Features com o Target (Risco)', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlação de Pearson')
plt.tight_layout()
plt.savefig('correlacao_features.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: correlacao_features.png')

---
## 5. Preparação para Treinamento

In [ ]:
# ─── 5.1 Seleção de Features ─────────────────────────────────────────────────
FEATURES = [
    'valor_transacao',
    'limite',
    'porcentagem_limit_usad',
    'hora',
    'flag_noturna',
    'flag_limite_alto',
    'flag_valor_alto',
    'flag_limite_zero',
    'ratio_valor_limite',
    'score_risco_bruto',
]

TARGET = 'target_num'

X = df[FEATURES]
y = df[TARGET]

print(f'✅ Features selecionadas ({len(FEATURES)}): {FEATURES}')
print(f'\nDistribuição do target:')
print(y.value_counts())

In [ ]:
# ─── 5.2 Divisão Treino / Teste ──────────────────────────────────────────────
# Estratificado para preservar a proporção de classes no split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'Treino: {X_train.shape[0]:,} amostras')
print(f'Teste:  {X_test.shape[0]:,} amostras')
print(f'\nProporção de risco no treino: {y_train.mean()*100:.3f}%')
print(f'Proporção de risco no teste:  {y_test.mean()*100:.3f}%')

---
## 6. Treinamento dos Modelos

> Utilizamos **SMOTE** (Synthetic Minority Oversampling Technique) dentro do pipeline para lidar com o desbalanceamento extremo (~0,15% de risco) **sem vazar dados do teste para o treino**.

In [ ]:
# ─── 6.1 Definição dos Pipelines ────────────────────────────────────────────
# Cada pipeline: Escalonamento → SMOTE (só no treino) → Modelo

pipeline_rf = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=SEED, sampling_strategy=0.2)),
    ('model',  RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ))
])

pipeline_gb = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=SEED, sampling_strategy=0.2)),
    ('model',  GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        random_state=SEED
    ))
])

print('✅ Pipelines definidos:')
print('   → Random Forest (RF)')
print('   → Gradient Boosting (GB)')

In [ ]:
# ─── 6.2 Treinamento — Random Forest ────────────────────────────────────────
print('🌲 Treinando Random Forest...')
pipeline_rf.fit(X_train, y_train)
print('✅ Random Forest treinado!')

In [ ]:
# ─── 6.3 Treinamento — Gradient Boosting ────────────────────────────────────
print('⚡ Treinando Gradient Boosting...')
pipeline_gb.fit(X_train, y_train)
print('✅ Gradient Boosting treinado!')

---
## 7. Avaliação dos Modelos

In [ ]:
# ─── 7.1 Função auxiliar de avaliação ───────────────────────────────────────
def avaliar_modelo(pipeline, X_test, y_test, nome_modelo, cor):
    """
    Avalia um pipeline treinado e retorna métricas + probabilidades.
    """
    y_pred      = pipeline.predict(X_test)
    y_prob      = pipeline.predict_proba(X_test)[:, 1]
    auc_roc     = roc_auc_score(y_test, y_prob)
    auc_pr      = average_precision_score(y_test, y_prob)
    
    print(f'\n{'='*55}')
    print(f'  📋 {nome_modelo}')
    print(f'{'='*55}')
    print(f'  AUC-ROC:              {auc_roc:.4f}')
    print(f'  AUC-PR (Avg Prec.):   {auc_pr:.4f}')
    print(f'\n  Relatório de Classificação:')
    print(classification_report(y_test, y_pred, target_names=['Sem Risco', 'Com Risco']))
    
    return y_pred, y_prob, auc_roc, auc_pr

In [ ]:
# ─── 7.2 Avaliação dos dois modelos ─────────────────────────────────────────
pred_rf, prob_rf, auc_rf, pr_rf = avaliar_modelo(
    pipeline_rf, X_test, y_test, 'Random Forest', '#27ae60'
)

pred_gb, prob_gb, auc_gb, pr_gb = avaliar_modelo(
    pipeline_gb, X_test, y_test, 'Gradient Boosting', '#2980b9'
)

In [ ]:
# ─── 7.3 Curvas ROC lado a lado ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelos = [
    ('Random Forest',     prob_rf, pred_rf, auc_rf, '#27ae60'),
    ('Gradient Boosting', prob_gb, pred_gb, auc_gb, '#2980b9'),
]

for ax, (nome, prob, pred, auc, cor) in zip(axes, modelos):
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=cor, lw=2.5, label=f'{nome}\nAUC = {auc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.5, label='Aleatório')
    ax.fill_between(fpr, tpr, alpha=0.1, color=cor)
    ax.set_xlabel('Taxa de Falsos Positivos')
    ax.set_ylabel('Taxa de Verdadeiros Positivos')
    ax.set_title(f'Curva ROC — {nome}', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.suptitle('Comparativo de Curvas ROC', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('curvas_roc.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: curvas_roc.png')

In [ ]:
# ─── 7.4 Matrizes de Confusão ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (nome, pred, cor) in zip(axes, [
    ('Random Forest', pred_rf, 'Greens'),
    ('Gradient Boosting', pred_gb, 'Blues')
]):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Sem Risco', 'Com Risco']
    )
    disp.plot(ax=ax, cmap=cor, colorbar=False)
    ax.set_title(f'Matriz de Confusão — {nome}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusão dos Modelos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('matrizes_confusao.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: matrizes_confusao.png')

In [ ]:
# ─── 7.5 Importância das Features (Random Forest) ───────────────────────────
importancias = pipeline_rf.named_steps['model'].feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importancia': importancias
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
cores_imp = ['#e74c3c' if 'flag' in f or 'score' in f or 'ratio' in f else '#3498db'
             for f in feat_imp_df['Feature']]
ax.barh(feat_imp_df['Feature'], feat_imp_df['Importancia'], color=cores_imp, edgecolor='white')
ax.set_title('Importância das Features — Random Forest\n(🔴 Features derivadas | 🔵 Features originais)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importância Relativa')
plt.tight_layout()
plt.savefig('importancia_features.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: importancia_features.png')

In [ ]:
# ─── 7.6 Tabela Comparativa Final ───────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

comparativo = pd.DataFrame([
    {
        'Modelo': 'Random Forest',
        'AUC-ROC': round(auc_rf, 4),
        'AUC-PR':  round(pr_rf, 4),
        'Precision (Risco)': round(precision_score(y_test, pred_rf), 4),
        'Recall (Risco)':    round(recall_score(y_test, pred_rf), 4),
        'F1 (Risco)':        round(f1_score(y_test, pred_rf), 4),
    },
    {
        'Modelo': 'Gradient Boosting',
        'AUC-ROC': round(auc_gb, 4),
        'AUC-PR':  round(pr_gb, 4),
        'Precision (Risco)': round(precision_score(y_test, pred_gb), 4),
        'Recall (Risco)':    round(recall_score(y_test, pred_gb), 4),
        'F1 (Risco)':        round(f1_score(y_test, pred_gb), 4),
    }
])

print('\n📊 COMPARATIVO FINAL DOS MODELOS')
print('=' * 75)
print(comparativo.to_string(index=False))
print('=' * 75)

---
## 8. Seleção do Melhor Modelo e Score de Risco

> **Critério de seleção:** Para detecção de risco financeiro, priorizamos **AUC-ROC** e **Recall** — queremos minimizar falsos negativos (transações de risco não detectadas), ainda que isso aumente um pouco os falsos positivos.

In [ ]:
# ─── 8.1 Seleção automática do melhor modelo ─────────────────────────────────
if auc_rf >= auc_gb:
    melhor_pipeline = pipeline_rf
    melhor_nome = 'Random Forest'
    melhor_prob = prob_rf
else:
    melhor_pipeline = pipeline_gb
    melhor_nome = 'Gradient Boosting'
    melhor_prob = prob_gb

print(f'🏆 Melhor modelo selecionado: {melhor_nome}')
print(f'   AUC-ROC: RF={auc_rf:.4f} | GB={auc_gb:.4f}')

In [ ]:
# ─── 8.2 Geração do Score de Risco para TODOS os dados ──────────────────────
# O score é a probabilidade estimada de ser uma transação de risco (0 a 1)
# Multiplicado por 100 → escala 0–100 para o dashboard

print(f'⏳ Gerando score de risco com o modelo {melhor_nome}...')

prob_completo = melhor_pipeline.predict_proba(X)[:, 1]

df['score_risco_modelo'] = (prob_completo * 100).round(2)

# Classificação do score em faixas (para facilitar o storytelling no Power BI)
def classificar_risco(score):
    if score >= 70:
        return 'CRÍTICO'
    elif score >= 40:
        return 'ALTO'
    elif score >= 20:
        return 'MODERADO'
    else:
        return 'BAIXO'

df['nivel_risco'] = df['score_risco_modelo'].apply(classificar_risco)

print('\nDistribuição dos níveis de risco:')
print(df['nivel_risco'].value_counts())
print(f'\nScore médio de risco: {df["score_risco_modelo"].mean():.2f}')
print(f'Score máximo:         {df["score_risco_modelo"].max():.2f}')
print('\n✅ Score gerado com sucesso!')

In [ ]:
# ─── 8.3 Visualização do Score de Risco ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma do score
axes[0].hist(df['score_risco_modelo'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(20, color='#f39c12', linestyle='--', lw=2, label='Limite BAIXO/MODERADO (20)')
axes[0].axvline(40, color='#e67e22', linestyle='--', lw=2, label='Limite MODERADO/ALTO (40)')
axes[0].axvline(70, color='#e74c3c', linestyle='--', lw=2, label='Limite ALTO/CRÍTICO (70)')
axes[0].set_title('Distribuição do Score de Risco', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score de Risco (0–100)')
axes[0].set_ylabel('Frequência')
axes[0].legend(fontsize=8)

# Distribuição dos níveis
nivel_counts = df['nivel_risco'].value_counts().reindex(['CRÍTICO', 'ALTO', 'MODERADO', 'BAIXO'])
cores_nivel = ['#c0392b', '#e74c3c', '#f39c12', '#27ae60']
axes[1].bar(nivel_counts.index, nivel_counts.values, color=cores_nivel, edgecolor='white')
for i, v in enumerate(nivel_counts.values):
    axes[1].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Transações por Nível de Risco', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nível de Risco')
axes[1].set_ylabel('Número de Transações')

plt.suptitle(f'Score de Risco — Modelo: {melhor_nome}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('score_risco_distribuicao.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Gráfico salvo: score_risco_distribuicao.png')

---
## 9. Exportação dos Resultados para o Power BI

> O CSV gerado aqui é a **entrada para a colega responsável pelo dashboard e storytelling**. Ele contém as variáveis originais + as features criadas + o score de risco do modelo.

In [ ]:
# ─── 9.1 Montagem do DataFrame de saída ──────────────────────────────────────
colunas_saida = [
    # Identificação
    'id_transacao',
    'data_trans',
    'hora',
    # Variáveis financeiras originais
    'valor_transacao',
    'limite',
    'porcentagem_limit_usad',
    # Features de risco
    'flag_noturna',
    'flag_limite_alto',
    'flag_valor_alto',
    'flag_limite_zero',
    'ratio_valor_limite',
    'score_risco_bruto',
    # Resultado do modelo
    'score_risco_modelo',
    'nivel_risco',
    # Target real (para validação)
    'target',
]

df_saida = df[colunas_saida].copy()
df_saida['data_trans'] = df_saida['data_trans'].astype(str)

print(f'📋 Linhas no arquivo de saída: {len(df_saida):,}')
print(f'📋 Colunas: {df_saida.columns.tolist()}')
print(f'\nAmostra das primeiras linhas:')
df_saida.head()

In [ ]:
# ─── 9.2 Exportação para CSV ─────────────────────────────────────────────────
ARQUIVO_SAIDA = 'risco_financeiro_com_score.csv'

df_saida.to_csv(ARQUIVO_SAIDA, index=False, encoding='utf-8-sig')

print(f'✅ Arquivo exportado com sucesso!')
print(f'   → {ARQUIVO_SAIDA}')
print(f'   → {len(df_saida):,} linhas × {len(df_saida.columns)} colunas')
print(f'\n📌 Este arquivo está pronto para importação no Power BI.')

In [ ]:
# ─── 9.3 Resumo Estatístico por Nível de Risco ───────────────────────────────
# Tabela útil para a colega usar como referência no storytelling
resumo_nivel = df_saida.groupby('nivel_risco').agg(
    total_transacoes=('id_transacao', 'count'),
    valor_medio=('valor_transacao', 'mean'),
    valor_total=('valor_transacao', 'sum'),
    score_medio=('score_risco_modelo', 'mean'),
    pct_risco_real=('target', lambda x: (x == 'Yes').mean() * 100)
).reset_index()

resumo_nivel['valor_medio'] = resumo_nivel['valor_medio'].round(2)
resumo_nivel['valor_total'] = resumo_nivel['valor_total'].round(2)
resumo_nivel['score_medio'] = resumo_nivel['score_medio'].round(2)
resumo_nivel['pct_risco_real'] = resumo_nivel['pct_risco_real'].round(4)

ordem = ['CRÍTICO', 'ALTO', 'MODERADO', 'BAIXO']
resumo_nivel = resumo_nivel.set_index('nivel_risco').reindex(ordem).reset_index()

resumo_nivel.to_csv('resumo_por_nivel_risco.csv', index=False, encoding='utf-8-sig')

print('📊 RESUMO POR NÍVEL DE RISCO')
print('=' * 80)
print(resumo_nivel.to_string(index=False))
print('\n💾 Arquivo salvo: resumo_por_nivel_risco.csv')

---
## 10. Conclusões e Entregáveis

### Resultados do Modelo

| Critério | Resultado |
|---|---|
| **Modelo selecionado** | Ver célula de seleção automática |
| **AUC-ROC** | Ver tabela comparativa |
| **Problema tratado** | Desbalanceamento extremo (~0,15% de risco) via SMOTE |
| **Features criadas** | 6 variáveis derivadas de engenharia de features |

### Arquivos Gerados

| Arquivo | Destino | Descrição |
|---|---|---|
| `risco_financeiro_com_score.csv` | **Power BI** | Dataset completo com score de risco por transação |
| `resumo_por_nivel_risco.csv` | **Power BI** | Agregado por nível para KPIs e cards |
| `distribuicao_target.png` | Apresentação | Visualização do desbalanceamento de classes |
| `distribuicao_variaveis.png` | Apresentação | KDE por classe de risco |
| `boxplot_variaveis.png` | Apresentação | Boxplot comparativo |
| `risco_por_hora.png` | Apresentação | Taxa de risco temporal |
| `correlacao_features.png` | Apresentação | Correlação das features com o target |
| `curvas_roc.png` | Apresentação | Curvas ROC dos modelos |
| `matrizes_confusao.png` | Apresentação | Matrizes de confusão |
| `importancia_features.png` | Apresentação | Importância das variáveis |
| `score_risco_distribuicao.png` | Apresentação | Distribuição do score gerado |

### Sugestão de KPIs para o Dashboard (Power BI)

- **Total de transações em risco** (nível ALTO + CRÍTICO)
- **Score médio de risco** da base
- **Valor financeiro exposto** (soma de `valor_transacao` dos níveis ALTO e CRÍTICO)
- **% de transações por nível** (CRÍTICO / ALTO / MODERADO / BAIXO)
- **Heatmap hora × nível de risco**
- **Evolução temporal do score médio de risco** (série temporal)